Mini Project

Title: Fine-Tuning Transformer for Sentiment Classification

Problem Statement:
Fine-tune a pretrained BERT model for sentiment classification using IMDb dataset.



1. **Install**



In [20]:
!pip install transformers datasets evaluate


2.  **Imports**



In [21]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate


3. **Check GPU**



In [22]:
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [23]:
dataset = load_dataset("imdb")

# Reduce size (important for fast training)
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(2000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(1000))

In [24]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [25]:
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [26]:
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [27]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [32]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.381296,0.879000
2,0.285489,0.379618,0.900000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.28548858642578123, metrics={'train_runtime': 489.6422, 'train_samples_per_second': 8.169, 'train_steps_per_second': 1.021, 'total_flos': 1052444221440000.0, 'train_loss': 0.28548858642578123, 'epoch': 2.0})

In [35]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.3796175420284271, 'eval_accuracy': 0.9, 'eval_runtime': 32.7201, 'eval_samples_per_second': 30.562, 'eval_steps_per_second': 3.82, 'epoch': 2.0}


In [37]:
text = "This movie was fantastic!"

# Move inputs to GPU
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Prediction
outputs = model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1).item()

print("Prediction:", "Positive" if prediction == 1 else "Negative")

Prediction: Positive
